T5 News Summarization Fine-Tuning (Seq2Seq Trainer)

In [1]:
# %pip install -q -U transformers datasets evaluate sentencepiece accelerate rouge_score nltk scikit-learn pandas


In [ ]:

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import nltk
import evaluate
import torch

from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

try:
    nltk.download("punkt", quiet=True)
except Exception:
    pass

os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "t5-small"
MAX_INPUT_LENGTH = 16
MAX_TARGET_LENGTH = 4


VAL_SIZE = 0.10

LEARNING_RATE = 2e-4
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
WEIGHT_DECAY = 0.01
NUM_TRAIN_EPOCHS = 3
EARLY_STOPPING_PATIENCE = 2

NUM_BEAMS = 4
NO_REPEAT_NGRAM_SIZE = 3
LENGTH_PENALTY = 1.0
EARLY_STOPPING_GENERATION = True

OUTPUT_DIR = "t5_news_summarization_output"

config_summary = pd.DataFrame(
    {
        "Setting": [
            "Model",
            "Max input length",
            "Max target length",
            "Validation split",
            "Learning rate",
            "Train batch size",
            "Eval batch size",
            "Gradient accumulation steps",
            "Weight decay",
            "Epochs",
            "Beam search (num_beams)",
            "No-repeat ngram size",
            "Length penalty",
            "Seed",
        ],
        "Value": [
            MODEL_NAME,
            MAX_INPUT_LENGTH,
            MAX_TARGET_LENGTH,
            VAL_SIZE,
            LEARNING_RATE,
            TRAIN_BATCH_SIZE,
            EVAL_BATCH_SIZE,
            GRADIENT_ACCUMULATION_STEPS,
            WEIGHT_DECAY,
            NUM_TRAIN_EPOCHS,
            NUM_BEAMS,
            NO_REPEAT_NGRAM_SIZE,
            LENGTH_PENALTY,
            SEED,
        ],
    }
)

config_summary


,Setting,Value
0,Model,t5-small
1,Max input length,16
2,Max target length,4
3,Validation split,0.1
4,Learning rate,0.0002
5,Train batch size,4
6,Eval batch size,4
7,Gradient accumulation steps,4
8,Weight decay,0.01
9,Epochs,3


In [3]:
TRAIN_CSV = Path("News_Summarization_train.csv")
TEST_CSV = Path("News_Summarization_test.csv")

if not TRAIN_CSV.exists() or not TEST_CSV.exists():
    raise FileNotFoundError(
        "Could not find News_Summarization_train.csv and/or News_Summarization_test.csv. "
        "Please place both files in the same folder as this notebook and run again."
    )

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

required_columns = {"article", "reference_summary"}

if not required_columns.issubset(train_df.columns):
    raise ValueError(
        f"Training file is missing required columns. Found: {list(train_df.columns)}"
    )

if not required_columns.issubset(test_df.columns):
    raise ValueError(
        f"Test file is missing required columns. Found: {list(test_df.columns)}"
    )

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    out = df[["article", "reference_summary"]].copy()
    out = out.dropna(subset=["article", "reference_summary"])
    out["article"] = (
        out["article"]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    out["reference_summary"] = (
        out["reference_summary"]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    out = out[(out["article"] != "") & (out["reference_summary"] != "")]
    return out.reset_index(drop=True)

train_df = clean_dataframe(train_df)
test_df = clean_dataframe(test_df)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

display(train_df.head(4))

Train shape: (2400, 2)
Test shape : (600, 2)


,article,reference_summary
0,"If you're like us, you eat out more than ever ...",Health magazine names Top 10 chain restaurants...
1,"LAGOS, Nigeria (Reuters) -- Nigeria's televisi...","Anthony Ogadje, 25, reportedly drowned in Sher..."
2,"ISLAMABAD, Pakistan (CNN) -- Suspected Taliban...","Blast targets boy's high school in Mingora, Sw..."
3,"LOS ANGELES, California (CNN) -- Farrah Fawcet...","NEW: Farrah Fawcett not unconscious, unrespons..."


In [4]:
data_stats = pd.DataFrame(
    {
        "Split": ["train", "test"],
        "Rows": [len(train_df), len(test_df)],
        "Avg article words": [
            round(train_df["article"].str.split().str.len().mean(), 2),
            round(test_df["article"].str.split().str.len().mean(), 2),
        ],
        "Avg summary words": [
            round(train_df["reference_summary"].str.split().str.len().mean(), 2),
            round(test_df["reference_summary"].str.split().str.len().mean(), 2),
        ],
    }
)

data_stats

,Split,Rows,Avg article words,Avg summary words
0,train,2400,606.37,43.46
1,test,600,603.08,43.50


In [5]:
train_split_df, val_df = train_test_split(
    train_df,
    test_size=VAL_SIZE,
    random_state=SEED,
    shuffle=True,
)

train_split_df = train_split_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

raw_datasets = DatasetDict(
    {
        "train": Dataset.from_pandas(train_split_df, preserve_index=False),
        "validation": Dataset.from_pandas(val_df, preserve_index=False),
        "test": Dataset.from_pandas(test_df, preserve_index=False),
    }
)

raw_datasets

DatasetDict({
    train: Dataset({
        features: ['article', 'reference_summary'],
        num_rows: 2160
    })
    validation: Dataset({
        features: ['article', 'reference_summary'],
        num_rows: 240
    })
    test: Dataset({
        features: ['article', 'reference_summary'],
        num_rows: 600
    })
})

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def preprocess_function(examples):
    inputs = ["summarize: " + article for article in examples["article"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        padding="max_length",
        truncation=True,
    )

    labels = tokenizer(
        text_target=examples["reference_summary"],
        max_length=MAX_TARGET_LENGTH,
        padding="max_length",
        truncation=True,
    )


    labels["input_ids"] = [
        [(token_id if token_id != tokenizer.pad_token_id else -100) for token_id in label_ids]
        for label_ids in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    desc="Tokenizing train/validation/test splits",
)

tokenized_datasets

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing train/validation/test splits:   0%|          | 0/2160 [00:00<?, ? examples/s]

Tokenizing train/validation/test splits:   0%|          | 0/240 [00:00<?, ? examples/s]

Tokenizing train/validation/test splits:   0%|          | 0/600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2160
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 240
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 600
    })
})

In [7]:
rouge = evaluate.load("rouge")

def _safe_sent_tokenize(text: str):
    try:
        return nltk.sent_tokenize(text)
    except LookupError:
        try:
            nltk.download("punkt", quiet=True)
            return nltk.sent_tokenize(text)
        except Exception:
            return [text]

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    preds = ["\n".join(_safe_sent_tokenize(pred)) for pred in preds]
    labels = ["\n".join(_safe_sent_tokenize(label)) for label in labels]

    return preds, labels

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )

    result = {k: float(v) for k, v in result.items()}

    pred_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = float(np.mean(pred_lens))

    return result

In [8]:
# !pip install -q transformers==4.41.2 datasets evaluate accelerate

In [9]:
import torch
print(torch.cuda.is_available())

False


In [10]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    do_train=True,
    do_eval=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=NUM_BEAMS,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="rouge1",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    fp16=torch.cuda.is_available(),
)

SMALL_SIZE = 5

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    # train_dataset=tokenized_datasets["train"],
    # eval_dataset=tokenized_datasets["validation"],
    train_dataset = tokenized_datasets["train"].select(range(SMALL_SIZE)),
    eval_dataset   = tokenized_datasets["validation"].select(range(5)),
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

train_result = trainer.train()
train_result

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,No log,7.678649,0.266667,0.200000,0.266667,0.266667,3.000000
2,No log,7.363748,0.266667,0.200000,0.266667,0.266667,3.000000
3,No log,7.253326,0.266667,0.200000,0.266667,0.266667,3.000000


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=3, training_loss=8.91386604309082, metrics={'train_runtime': 28.8573, 'train_samples_per_second': 0.52, 'train_steps_per_second': 0.104, 'total_flos': 63441469440.0, 'train_loss': 8.91386604309082, 'epoch': 3.0})

In [11]:
test_prediction_output = trainer.predict(
    tokenized_datasets["test"],
    metric_key_prefix="test",
)

generated_token_ids = test_prediction_output.predictions
generated_summaries = tokenizer.batch_decode(
    generated_token_ids,
    skip_special_tokens=True,
)
generated_summaries = [summary.strip() for summary in generated_summaries]

test_results_df = test_df.copy()
test_results_df["generated_summary"] = generated_summaries

final_rouge = rouge.compute(
    predictions=test_results_df["generated_summary"].tolist(),
    references=test_results_df["reference_summary"].tolist(),
    use_stemmer=True,
)

final_rouge = {k: float(v) for k, v in final_rouge.items()}

print("Final test ROUGE scores:")
for metric_name, metric_value in final_rouge.items():
    print(f"{metric_name}: {metric_value:.4f}")

print(f"\nRequired score -> Test ROUGE-1 F1: {final_rouge['rouge1']:.4f}")

test_results_df.to_csv("problem2_test_predictions.csv", index=False)
print("\nSaved predictions to: problem2_test_predictions.csv")


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Final test ROUGE scores:
rouge1: 0.0384
rouge2: 0.0072
rougeL: 0.0363
rougeLsum: 0.0364

Required score -> Test ROUGE-1 F1: 0.0384

Saved predictions to: problem2_test_predictions.csv


In [12]:

summary_scores = pd.DataFrame(
    {
        "Metric": ["ROUGE-1 F1", "ROUGE-2 F1", "ROUGE-L F1", "ROUGE-Lsum F1"],
        "Score": [
            final_rouge.get("rouge1", np.nan),
            final_rouge.get("rouge2", np.nan),
            final_rouge.get("rougeL", np.nan),
            final_rouge.get("rougeLsum", np.nan),
        ],
    }
)

summary_scores.style.format({"Score": "{:.4f}"})

,Metric,Score
0,ROUGE-1 F1,0.0384
1,ROUGE-2 F1,0.0072
2,ROUGE-L F1,0.0363
3,ROUGE-Lsum F1,0.0364


In [13]:
example_rows = test_results_df.sample(n=3, random_state=SEED).reset_index(drop=True)

display(example_rows[["article", "reference_summary", "generated_summary"]])

,article,reference_summary,generated_summary
0,"LOS ANGELES, California (CNN) -- This year, th...",Hugh Jackman hosts Oscars on Sunday night . Au...,LOS
1,(CNN) -- The International Committee of the Re...,NEW: Defense minister blasts leak of video fro...,the international committee
2,WASHINGTON (CNN) -- The House of Representativ...,President Obama has promised to sign Lilly Led...,the house of


In [14]:
def auto_quality_comment(reference: str, prediction: str) -> str:
    score = rouge.compute(
        predictions=[prediction],
        references=[reference],
        use_stemmer=True,
    )["rouge1"]

    pred_len = len(prediction.split())
    ref_len = max(len(reference.split()), 1)
    length_ratio = pred_len / ref_len

    if score >= 0.50:
        overlap_desc = "strong content overlap with the reference summary"
    elif score >= 0.40:
        overlap_desc = "good overlap, with only minor omissions or paraphrasing differences"
    elif score >= 0.30:
        overlap_desc = "moderate overlap; the main point is partly captured, but some details may be missing"
    else:
        overlap_desc = "limited lexical overlap; manual inspection is needed to check whether the difference is due to paraphrasing or missed facts"

    if length_ratio < 0.80:
        len_desc = "more concise than the reference"
    elif length_ratio > 1.20:
        len_desc = "longer than the reference"
    else:
        len_desc = "roughly similar in length to the reference"

    return f"ROUGE-1={score:.3f}; the generated summary shows {overlap_desc} and is {len_desc}."

for i, row in example_rows.iterrows():
    print("=" * 100)
    print(f"Example {i+1}")
    print("-" * 100)

    article_text = row["article"]
    article_preview = article_text if len(article_text) <= 1500 else article_text[:1500] + " ..."

    print("Article preview:")
    print(article_preview)
    print("\nReference summary:")
    print(row["reference_summary"])
    print("\nModel-generated summary:")
    print(row["generated_summary"])
    print("\nBrief quality comment:")
    print(auto_quality_comment(row["reference_summary"], row["generated_summary"]))
    print()

Example 1
----------------------------------------------------------------------------------------------------
Article preview:
LOS ANGELES, California (CNN) -- This year, the Oscars are on Hugh Jackman's shoulders. Hugh Jackman plans to offer viewers "a good time" at the Oscars. He hosts the big show Sunday night. The Australian actor, who earned rave reviews for his hosting of the Tonys, now has the Academy Awards to contend with. It's a job that's put Jon Stewart, Chris Rock, Whoopi Goldberg and David Letterman on the firing line, with only Billy Crystal and Johnny Carson emerging more or less unscathed in the last couple of decades. But with typical verve -- after all, this is the guy who won a Tony for playing song-and-dance man Peter Allen in "The Boy from Oz" -- Jackman cracks jokes about the task, telling ABC that one of his distinctions is that he's the "tallest" Oscar host in recent years. To CNN's Brooke Anderson, he was equally at ease. "Ultimately, the way I see it is if I

In [15]:
print(f"Final reported test ROUGE-1 F1: {final_rouge['rouge1']:.4f}")

if final_rouge["rouge1"] >= 0.4000:
    print("Target met: ROUGE-1 F1 is at least 0.4000.")
else:
    print("Target not yet met. Suggested next adjustments: train for more epochs, reduce the learning rate to 1e-4, or increase num_beams to 5 or 6.")

Final reported test ROUGE-1 F1: 0.0384
Target not yet met. Suggested next adjustments: train for more epochs, reduce the learning rate to 1e-4, or increase num_beams to 5 or 6.
